# CREDTIA — Train / Test avec XGBoost

Cette notebook utilise une **base synthétique** construite selon le schéma CREDTIA V1. Elle sert uniquement à valider techniquement le pipeline. La cible `default_90d` est simulée et devra être remplacée par des défauts réellement observés.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, confusion_matrix
from xgboost import XGBClassifier

df = pd.read_csv("CREDTIA_Historique_Synthetique_TrainTest.csv")
print(df.shape)
df.head()


## 1. Vérifier la cible et la variable cotisation

In [ ]:
print(df["default_90d"].value_counts(normalize=True))
print("\nCotisations à jour :")
print(df["cotisations_a_jour"].value_counts())

# Contrôle métier : les dossiers non à jour sont bloqués.
print("\nDécision par statut cotisation :")
print(pd.crosstab(df["cotisations_a_jour"], df["decision_final"]))


## 2. Construire X et y

On retire les variables connues seulement après la décision, ainsi que les sorties de l'ancien moteur expert, afin d'éviter la fuite d'information.

In [ ]:
TARGET = "default_90d"

LEAKAGE_OR_POST_DECISION = [
    "default_30d", "default_90d", "days_past_due_max", "total_repaid",
    "observation_complete", "decision_final", "approved_amount",
    "override_flag", "override_reason",
    "expert_score", "risk_class", "data_quality_score",
    "max_installment", "qualitative_score", "quantitative_score",
    "mandatory_block"
]

DROP_COLS = [c for c in LEAKAGE_OR_POST_DECISION if c in df.columns] + ["application_id", "analysis_date"]

X = df.drop(columns=DROP_COLS + [TARGET], errors="ignore")
y = df[TARGET]

print("X:", X.shape, "y:", y.shape)


## 3. Train / Test

Pour une vraie étude CREDTIA, il faudra privilégier un **split temporel** : anciennes demandes pour l'entraînement et demandes plus récentes pour le test.

In [ ]:
# Démonstration technique
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)


## 4. Prétraitement + XGBoost

In [ ]:
categorical = X_train.select_dtypes(include=["object"]).columns.tolist()
numeric = X_train.select_dtypes(exclude=["object"]).columns.tolist()

preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical)
])

pos = max((y_train == 0).sum() / max((y_train == 1).sum(), 1), 1)

model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=pos,
    random_state=42
)

pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", model)
])

pipe.fit(X_train, y_train)


## 5. Évaluer le modèle

In [ ]:
proba = pipe.predict_proba(X_test)[:,1]
pred = (proba >= 0.50).astype(int)

print("ROC-AUC :", round(roc_auc_score(y_test, proba), 4))
print("PR-AUC  :", round(average_precision_score(y_test, proba), 4))
print("\nMatrice de confusion :")
print(confusion_matrix(y_test, pred))
print("\nClassification report :")
print(classification_report(y_test, pred))


## 6. Règle métier CREDTIA + ML

Le ML ne remplace pas les contrôles obligatoires. Une demande avec `cotisations_a_jour = Non` reste bloquée, même si la probabilité de défaut prédite est faible.

In [ ]:
def credtia_decision(row, predicted_pd):
    # Hard stop métier
    mandatory = (
        row["cotisations_a_jour"] == "Non"
        or row["kyc_complete"] == "Non"
        or row["bic_consent"] == "Non"
        or row["active_arrears_gt30d"] == "Oui"
        or row["fraud_major_inconsistency"] == "Oui"
        or row["purpose_authorized"] == "Non"
        or row["age_term_compliant"] == "Non"
    )
    if mandatory:
        return "BLOQUE"
    # Exemple de seuil ML provisoire pour démonstration uniquement
    if predicted_pd >= 0.50:
        return "RISQUE_ELEVE_REVUE"
    return "POURSUITE_VALIDATION"

results = df.loc[X_test.index, [
    "application_id", "cotisations_a_jour", "amount_requested"
]].copy()
results["predicted_pd"] = proba
results["ml_decision"] = [
    credtia_decision(df.loc[i], p) for i,p in zip(X_test.index, proba)
]
results.head(20)


## 7. Prochaine étape

Cette base est volontairement synthétique. Pour le vrai modèle CREDTIA, il faudra importer les historiques réels et renseigner `default_90d` à partir d'un événement défini objectivement (par exemple un seuil de jours de retard). Ensuite, on pourra comparer **score expert vs régression logistique vs Random Forest vs XGBoost**, puis faire la calibration et SHAP.